# Transformer 기본개념 2

## Mulit head Attention

먼저 트랜스포머 구조에서 'Multi head attention' 박스의 구조를 보자면,

  1. 입력에서 Q, K, V 생성                                                                                                                                    
  2. 여러 head로 나눔                                                                                                                                         
  3. 각 head가 attention 수행                                                                                                                                 
  4. head 결과들을 Concat                                                                                                                                     
  5. W_O로 선형변환                                                                                                                                           
  6. 그 결과가 Multi-Head Attention 블록의 출력이 됨                                                                                                          
  7. 그 다음에 Add & Norm으로 넘어감

## Mask

mask는 학습할 떄에만 적용되고 실제 기계 번역에서는 적용되지 않을까?
> No!!

디코더의 mask는 학습할 때도 쓰이고 실제 생성할 때도 필요하다

학습 때는 mask를 통해 디코더가 각 위치 자기 앞까지만 보게 한다

실제 번역(다음 단어 추론) 할 때에는 보통 디코더가 단어를 한 개씩 순차적으로 생성하는데, 

  1. 시작 토큰 입력                                                                                                                                           
  2. 첫 단어 생성                                                                                                                                             
  3. 지금까지 생성된 단어들을 넣어서 다음 단어 생성                                                                                                           
  4. 반복  

이 경우 아직 미래 단어 자체가 존재하지 않으니까, 추론 때에도 미래를 보지 않는 제약은 그대로 유지된다. 다만, 추론은 단어를 하나씩 생성하므로 미래 단어가 애초에 입력이 없기 때문에 원리적으로는 필요 없지만, 형식 상으로만 존재한다.  

> 추론에서도 디코더의 각 위치는 자기까지의 이전 토큰들만 반영해서 표현이 만들어진다. 
>
> mask는 학습 전용 전략이다. 이를 통해 RNN 처럼 단어를 하나씩 순서대로 처리하는 하지 않으면서도 한 번에 문장을 넣고 뒤의 단어를 mask한 것이다 -> 순차적인 RNN의 한계를 극복하면서도 정답을 미리 보는 반칙을 막음

## 인코더와 디코더의 embedding

인코더 - input emdedding  
디코더 - output emdedding 

각각 다른 임베딩이 들어간다. 

1. 인코더 임베딩의 정체
> 입력 언어의 단어들을 벡터로 만든다.  
> ex 한국어를 벡터로 만든다.

2. 디코더 임베딩의 정체
> 출력 언어의 단어들을 벡터로 만든다.  
> ex 영어를 벡터로 만든다.


**이때 디코더는 빙글빙글 돈다(자기회귀)**

트랜스포머는 문장을 한 번에 만들어서 뱉는게 아니라 하나 생성하고 다시 디코더에 입력으로 집어넣는 루프를 가진다. -> **디코더의 임베딩 정체가 바로 그것이다**

1회차: 디코더에 <sos>(시작 토큰)만 넣습니다. ->  결과: "I" 예측

2회차: 디코더에 [<sos>, "I"]를 넣습니다 -> 결과: "am" 예측


3회차: 디코더에 [<sos>, "I", "am"]을 넣습니다. -> 결과: "a" 예측

4회차: 디코더에 [<sos>, "I", "am", "a"]를 넣습니다. -> 결과: "student" 예측

[<sos>, "I", "am", "a", "student"]를 넣었더니 결과로 <eos>(종료 토큰)이 나오면 루프를 멈춥니다.

*물론, 이것은 실제 추론할 때의 과정이다. 학습할 때에는 모든 정보를 한 번에 밀어넣고(+ mask) 학습하기에 속도가 압도적으로 빠르다. 


**중요한 것은 빙글빙글 도는 것은 오직 디코더이다!!**
> 인코더는 한 번만 일한다. '나는 학생입니다'라는 문장을 받으면, 단 한번의 계산으로 모든 단어 사이의 관계를 파악해 정보 보따리 (K,V)를 만들어둔다
>
> 디코더: 인코더가 만들어 놓은 보따리를 매 회차 계속 들처보면서 다음에 올 단어를 하나씩 생성한다. 


## 인코더 디코더의 Attention (Encoder-Decoder Cross-Attention)

디코더가 i am 까지 단어를 만들었다고 가정해보자

디코더는 i am 다음에 올 단어를 찾기 위해 현재 자신의 상태를 Q로 만든다. 그리고 인코더가 준 K와 내적을 한다. -> 역서 attention socre를 만든다. -> 점수가 가장 높은 학생의 Attention Value를 비중있게 가져오게 된다!


Attention Score (어텐션 스코어): Query와 Key를 내적한 값입니다 ($Q \cdot K^T$).

Attention Value (어텐션 밸류): 입력된 단어들의 의미를 담은 벡터V 그 자체입니다.